# 11 · Single stocks and ETFs: AAPL, NVDA, TSLA, QQQ and SLV, 2016–2023

**In the story** (notebook 00, §4.5 and §6): among the stocks and ETFs, AAPL gains a little; NVDA, TSLA and SLV gain nothing or lose; and QQQ, a Nasdaq-100 index ETF, behaves like SPX. Two failure modes appear here too: a crash entering the training window, followed by over-correction (the stocks' 2020), and a fixed scaling of delta applied outside the market it was read from (Ruf and Wang's 0.9/1.1 on single stocks).

Plan §16.15 applies the SPY protocol to five more underlyings. It was specified before any of their outcomes was computed. The analysis is exploratory, with no confirmatory verdict. Yearly and episode intervals are pointwise, without multiplicity adjustment.

The data are OptionsDX end-of-day files in the same format as SPX and SPY, with our own implied volatilities and Greeks. The inputs and methods are:

- puts from Lecture 12's American tree;
- calls from European Black–Scholes with the dividend yield;
- rates from SPX's parity curve;
- dividends from Nasdaq's history (TSLA and SLV pay none).

The level is the vendor's snapshot. Pairs straddling a stock split are excluded. The rules are R1 (Black–Scholes), R2 (Hull–White), R3 (Ruf–Wang fixed scaling), R4 (Ruf–Wang fitted scaling), R6 (delta–vega–vanna), R9 (LightGBM on δ) and P0 (a constant correction).

For context, Hull and White's Table 3 (2004–2015, 36-month window) gives an average Gain over the 30 Dow stocks of 10.3% for calls and 2.5% for puts, and 7.7% and 2.5% for three commodity ETFs, one of them SLV.

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
R = ROOT / "results"
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})
U = [u for u in ("aapl", "nvda", "tsla", "qqq", "slv") if (R / "stocks" / u / "walkforward_gains_by_year.csv").exists()]
U

['aapl', 'nvda', 'tsla', 'qqq', 'slv']

## Data audit and attrition

In [2]:
for u in U:
    for f in sorted((R / "stocks" / u).glob("audit*.csv")) + sorted((R / "stocks" / u).glob("attrition*.csv")):
        print(u.upper(), f.name); display(pd.read_csv(f).head(20))
    s = R / "stocks" / u / "splits.csv"
    if s.exists():
        print(u.upper(), "splits detected"); display(pd.read_csv(s))

AAPL attrition_by_status.csv


,year,cp,next_status,pairs
0,2016,C,contract_absent,38
1,2016,C,ok,54592
2,2016,C,one_sided_next,55
3,2016,C,vendor_missing_session,226
4,2016,P,contract_absent,48
5,2016,P,ok,51556
6,2016,P,one_sided_next,68
7,2016,P,vendor_missing_session,206
8,2017,C,ok,55504
9,2017,C,one_sided_next,44


AAPL splits detected


,date,prev_date,move,ratio
0,2020-08-31,2020-08-28,-0.742083,4.0


NVDA attrition_by_status.csv


,year,cp,next_status,pairs
0,2016,C,ok,26188
1,2016,C,one_sided_next,27
2,2016,C,vendor_missing_session,84
3,2016,P,contract_absent,1
4,2016,P,ok,21601
5,2016,P,one_sided_next,97
6,2016,P,vendor_missing_session,59
7,2017,C,ok,51109
8,2017,C,one_sided_next,46
9,2017,C,vendor_missing_session,217


... (10 outputs left out)

SLV attrition_by_status.csv


,year,cp,next_status,pairs
0,2016,C,ok,31908
1,2016,C,one_sided_next,172
2,2016,C,vendor_missing_session,130
3,2016,P,ok,17515
4,2016,P,one_sided_next,82
5,2016,P,vendor_missing_session,62
6,2017,C,ok,24704
7,2017,C,one_sided_next,196
8,2017,C,vendor_missing_session,98
9,2017,P,ok,14613


SLV splits detected


,date,prev_date,move,ratio


## 2020 as the test year (2019 training, 12-month window), under the verdict procedure of §8

In [3]:
rows = []
for u in U:
    v = pd.read_csv(R / "stocks" / u / "test2020_verdict.csv").assign(underlying=u.upper())
    rows.append(v)
pd.concat(rows)[["underlying", "variant", "verdict", "G_C", "G_C_lo", "G_C_hi", "G_P", "G_P_lo", "G_P_hi", "C_vs_R3", "C_vs_R4", "P_vs_R3", "P_vs_R4"]].round(3)

,underlying,variant,verdict,G_C,G_C_lo,G_C_hi,G_P,G_P_lo,G_P_hi,C_vs_R3,C_vs_R4,P_vs_R3,P_vs_R4
0,AAPL,primary,Inconclusive,0.044,-0.370,0.244,0.007,-0.159,0.110,not distinguished,not distinguished,R2 beats,not distinguished
1,AAPL,W36,Inconclusive,0.060,-0.284,0.225,-0.001,-0.163,0.102,not distinguished,not distinguished,R2 beats,not distinguished
2,AAPL,F,Inconclusive,0.112,-0.153,0.249,-0.006,-0.180,0.098,R2 beats,not distinguished,R2 beats,not distinguished
3,AAPL,S3,Inconclusive,-0.048,-0.293,0.127,-0.014,-0.095,0.079,not distinguished,not distinguished,R2 beats,not distinguished
4,AAPL,V,Inconclusive,0.046,-0.384,0.245,0.007,-0.161,0.110,not distinguished,not distinguished,R2 beats,not distinguished
5,AAPL,rest of 2020,Reject (a),-0.297,-0.598,-0.035,-0.087,-0.251,0.047,not distinguished,not distinguished,not distinguished,R2 worse
0,NVDA,primary,Reject (b),-0.025,-0.207,0.040,-0.024,-0.098,0.023,not distinguished,not distinguished,not distinguished,not distinguished
1,NVDA,W36,Inconclusive,0.070,-0.153,0.180,0.041,-0.100,0.121,not distinguished,not distinguished,not distinguished,not distinguished
2,NVDA,F,Inconclusive,0.072,-0.008,0.123,0.043,-0.016,0.091,not distinguished,not distinguished,not distinguished,not distinguished
3,NVDA,S3,Reject (b),-0.094,-0.278,0.003,-0.032,-0.107,0.028,not distinguished,R2 beats,not distinguished,not distinguished


**Reading.** None of the five passes H1 in 2020 (H1 requires the 95% lower bound of R2's Gain over Black–Scholes to exceed 0 for calls and puts). AAPL, TSLA and QQQ are Inconclusive. NVDA is Reject (b): both upper bounds are below the 0.05 the plan calls material. SLV is Reject (a): its put interval lies below zero, so R2 is significantly worse than Black–Scholes.

Outside the stress period, AAPL's, NVDA's and QQQ's calls lose significantly. That is consistent with SPX's April 2020 failure, the crash entering the training window and the fits over-correcting afterwards. No monthly evidence is shown here, though, and on SPX itself the rest of 2020 gained (calls +0.126).

## The walk-forward: Gain over Black–Scholes by year

In [4]:
for u in U:
    G = pd.read_csv(R / "stocks" / u / "walkforward_gains_by_year.csv")
    x = G[(G.window == 12) & (G.base == "R1") & G.rule.isin(["R2", "R3", "R4", "R6", "R9", "P0"])]
    print(u.upper(), "W12, Gain over R1")
    display(x.pivot_table(index="year", columns=["rule", "cp"], values="gain").round(3))

AAPL W12, Gain over R1


rule     P0            R2            R3            R4            R6            R9       
cp        C      P      C      P      C      P      C      P      C      P      C      P
year                                                                                    
2017 -0.054 -0.039 -0.048 -0.024 -0.278 -0.062 -0.036 -0.017 -0.045 -0.022 -0.045 -0.025
2018  0.095  0.119  0.112  0.133 -0.052  0.130  0.021  0.127  0.090  0.128  0.106  0.131
2019  0.109  0.106  0.106  0.118 -0.018  0.093  0.043  0.115  0.077  0.117  0.112  0.118
2020  0.047  0.021  0.044  0.007 -0.001 -0.068 -0.012  0.007  0.030 -0.022  0.048  0.015
2021  0.014  0.011  0.031  0.015 -0.171 -0.017 -0.010  0.019  0.025  0.014  0.026  0.016
2022  0.113  0.023  0.121  0.023 -0.013 -0.034  0.047  0.029  0.123  0.024  0.123  0.028
2023  0.002 -0.017  0.011 -0.014 -0.181 -0.105 -0.011 -0.009  0.004 -0.016  0.008 -0.014

NVDA

 W12, Gain over R1


rule     P0            R2            R3            R4            R6            R9       
cp        C      P      C      P      C      P      C      P      C      P      C      P
year                                                                                    
2017 -0.308 -0.172 -0.317 -0.183 -0.221 -0.219 -0.201 -0.217 -0.305 -0.222 -0.288 -0.170
2018  0.232  0.231  0.232  0.231  0.227  0.257  0.160  0.197  0.206  0.216  0.233  0.232
2019 -0.121 -0.146 -0.120 -0.142 -0.192 -0.143 -0.083 -0.084 -0.107 -0.107 -0.120 -0.142
2020 -0.013 -0.022 -0.025 -0.024  0.063  0.013 -0.040 -0.014 -0.026 -0.029 -0.017 -0.023
2021 -0.023 -0.024 -0.022 -0.024 -0.305 -0.270 -0.005 -0.023 -0.017 -0.010 -0.022 -0.023
2022 -0.016 -0.098 -0.004 -0.093  0.006 -0.070 -0.026 -0.063 -0.003 -0.103 -0.009 -0.094
2023 -0.171 -0.089 -0.178 -0.096 -0.530 -0.483 -0.107 -0.113 -0.185 -0.106 -0.173 -0.092

TSLA

... (3 outputs left out)

rule     P0            R2            R3            R4            R6            R9       
cp        C      P      C      P      C      P      C      P      C      P      C      P
year                                                                                    
2017  0.089  0.086  0.092  0.086  0.069  0.066  0.089  0.071  0.091  0.090  0.091  0.088
2018  0.090  0.118  0.094  0.119  0.118  0.104  0.091  0.125  0.091  0.141  0.093  0.120
2019  0.098  0.068  0.099  0.059  0.067  0.008  0.086  0.043  0.098  0.065  0.100  0.066
2020  0.101  0.075  0.083  0.056  0.118  0.010  0.020  0.028  0.069  0.021  0.095  0.068
2021  0.065  0.084  0.070  0.064 -0.055  0.101  0.035  0.106  0.061  0.067  0.068  0.078
2022  0.044  0.038  0.047  0.039 -0.102 -0.042  0.024  0.038  0.053  0.040  0.049  0.044
2023 -0.084 -0.111 -0.080 -0.108 -0.429 -0.363 -0.065 -0.094 -0.081 -0.102 -0.081 -0.108

SLV

 W12, Gain over R1


rule     P0            R2            R3            R4            R6            R9       
cp        C      P      C      P      C      P      C      P      C      P      C      P
year                                                                                    
2017  0.003 -0.056  0.022 -0.040 -0.104 -0.122 -0.023 -0.027  0.029 -0.041  0.012 -0.050
2018  0.000 -0.000  0.002  0.002 -0.018 -0.007 -0.002  0.003  0.002  0.002  0.002  0.002
2019 -0.030 -0.011 -0.030 -0.010 -0.122 -0.120 -0.019 -0.009 -0.026 -0.001 -0.029 -0.010
2020 -0.118 -0.157 -0.121 -0.160 -0.053 -0.071 -0.100 -0.081 -0.116 -0.148 -0.120 -0.159
2021 -0.068 -0.043 -0.067 -0.043 -0.250 -0.263 -0.041 -0.033 -0.075 -0.038 -0.067 -0.043
2022 -0.002  0.016 -0.000  0.032 -0.106 -0.182 -0.000 -0.001  0.010  0.028  0.002  0.030
2023  0.005 -0.003  0.006  0.001 -0.054 -0.182  0.000 -0.005  0.015 -0.005  0.005  0.004

In [5]:
med = []
for u in U:
    m = pd.read_csv(R / "stocks" / u / "walkforward_median_yearly_gain.csv").assign(underlying=u.upper())
    med.append(m[(m.base == "R1") & m.rule.isin(["R2", "R3", "R4", "R9", "P0"])])
pd.concat(med).pivot_table(index=["underlying", "window"], columns=["rule", "cp"], values="median_yearly_gain").round(3)

rule                  P0            R2            R3            R4            R9       
cp                     C      P      C      P      C      P      C      P      C      P
underlying window                                                                      
AAPL       12      0.047  0.021  0.044  0.015 -0.052 -0.034 -0.010  0.019  0.048  0.016
           36      0.063  0.018  0.060  0.013 -0.018 -0.034  0.006  0.028  0.064  0.012
           60      0.007  0.010  0.023  0.016 -0.171 -0.034 -0.013  0.027  0.016  0.014
NVDA       12     -0.023 -0.089 -0.025 -0.093 -0.192 -0.143 -0.040 -0.063 -0.022 -0.092
           36     -0.000  0.008 -0.007 -0.008 -0.192 -0.143  0.009  0.015 -0.001  0.006
           60     -0.053 -0.075 -0.060 -0.076 -0.305 -0.270 -0.056 -0.056 -0.055 -0.075
QQQ        12      0.089  0.075  0.083  0.059  0.067  0.010  0.035  0.043  0.091  0.068
           36      0.056  0.070  0.055  0.056 -0.055  0.008  0.006  0.046  0.056  0.064
           60     -0.096  0.014 -0.087 -0.000 -0.102 -0.042 -0.009  0.056 -0.091  0.009
SLV        12     -0.002 -0.011 -0.000 -0.010 -0.104 -0.122 -0.019 -0.009  0.002 -0.010
           36     -0.030 -0.021 -0.030 -0.020 -0.106 -0.182 -0.021 -0.022 -0.029 -0.020
           60     -0.001  0.006  0.002  0.012 -0.106 -0.182 -0.002  0.001  0.001  0.010
TSLA       12     -0.046 -0.038 -0.046 -0.034 -0.372 -0.223 -0.030 -0.027 -0.045 -0.033
           36     -0.009 -0.016 -0.008 -0.013 -0.386 -0.223 -0.009 -0.015 -0.008 -0.010
           60      0.010  0.018  0.012  0.013 -0.422 -0.267  0.014 -0.001  0.013  0.022

**Reading.** The single stocks gain little or lose. The medians cover 2017–2023, 2019–2023 and 2021–2023 for the 12-, 36- and 60-month fits, respectively. AAPL's median yearly Gain of R2 is 0.04–0.06 for calls with 12- and 36-month fits (0.023 with 60) and at most 0.02 for puts. NVDA and TSLA are near zero or negative.

QQQ, an index ETF, behaves like SPX: 0.06–0.08 on the 12- and 36-month windows. SLV gains nothing.

Ruf and Wang's fixed 0.9/1.1 scaling (R3), calibrated to S&P 500 options, is harmful off the index: −0.05 to −0.37 for calls on the stocks with 12-month fits (−0.02 to −0.42 across the 12-, 36- and 60-month windows).

## Beside Hull and White's Table 3 (mean of monthly Gains, 36-month window)

In [6]:
hw = {"C": {"Stocks": 0.103, "Commodity ETFs": 0.077}, "P": {"Stocks": 0.025, "Commodity ETFs": 0.025}}
rows = []
for u in U:
    t = pd.read_csv(R / "stocks" / u / "hw_table3_comparison.csv")
    a = t[t.bucket.astype(str) == "all"].set_index("cp").gain
    rows.append({"underlying": u.upper(), "calls (ours)": a.get("C"), "puts (ours)": a.get("P")})
display(pd.DataFrame(rows).round(3)); hw

,underlying,calls (ours),puts (ours)
0,AAPL,0.012,0.005
1,NVDA,-0.054,-0.060
2,TSLA,-0.014,-0.001
3,QQQ,-0.081,-0.060
4,SLV,-0.033,-0.039


{'C': {'Stocks': 0.103, 'Commodity ETFs': 0.077},
 'P': {'Stocks': 0.025, 'Commodity ETFs': 0.025}}

**Reading.** Using Hull and White's kind of aggregation gives our stocks far less than their Dow-stock average of 0.103 (calls) and 0.025 (puts). The direction matches theirs: much less than for indices, and almost nothing for puts.

Their explanation fits: noise inserts a wedge between the parameters estimated in one period and those that would hedge best in the next.

QQQ's row needs care. The monthly average can be negative even when the pooled yearly Gains are positive, because single months with large losses weigh as much as calm months.

## Stress episodes, parameter stability, level and shape, maturity

In [7]:
for u in U:
    S = pd.read_csv(R / "stocks" / u / "stress_gains.csv")
    x = S[(S.window == 12) & (S.rule == "R2") & (S.base == "R1")]
    print(u.upper(), "stress episodes, R2 over R1 (W12)")
    display(x[["definition", "start", "end", "cp", "gain", "lo", "hi", "sessions", "best_constant", "fitted_constant"]].round(3))

AAPL stress episodes, R2 over R1 (W12)


,definition,start,end,cp,gain,lo,hi,sessions,best_constant,fitted_constant
1,RV,2018-02-12,2018-03-05,C,0.013,-0.081,0.082,15,-0.030,-0.052
13,RV,2018-02-12,2018-03-05,P,-0.068,-0.230,0.111,15,0.043,-0.054
25,RV,2018-12-26,2019-01-23,C,0.270,-0.061,0.407,19,-0.264,-0.137
37,RV,2018-12-26,2019-01-23,P,0.284,0.017,0.415,19,-0.249,-0.139
49,RV,2020-02-27,2020-05-21,C,0.260,-0.233,0.391,60,-0.274,-0.176
61,RV,2020-02-27,2020-05-21,P,0.059,-0.155,0.186,60,-0.120,-0.130
73,RV,2020-06-11,2020-07-09,C,-0.159,-0.829,0.316,18,-0.076,-0.254
85,RV,2020-06-11,2020-07-09,P,-0.217,-0.535,0.007,18,0.036,-0.123
97,RV,2022-03-09,2022-03-22,C,0.168,0.032,0.279,10,-0.181,-0.078
109,RV,2022-03-09,2022-03-22,P,0.196,0.090,0.281,10,-0.179,-0.073


NVDA

 stress episodes, R2 over R1 (W12)


,definition,start,end,cp,gain,lo,hi,sessions,best_constant,fitted_constant
1,RV,2018-02-12,2018-03-05,C,-0.200,-0.276,-0.128,15,-0.213,0.057
13,RV,2018-02-12,2018-03-05,P,-0.314,-0.426,-0.195,15,-0.186,0.084
25,RV,2018-12-26,2019-01-23,C,-0.368,-0.853,0.074,19,0.015,-0.195
37,RV,2018-12-26,2019-01-23,P,-0.312,-1.105,0.178,19,0.001,-0.186
49,RV,2020-02-27,2020-05-21,C,0.019,-0.253,0.073,60,-0.218,-0.067
61,RV,2020-02-27,2020-05-21,P,-0.035,-0.209,-0.004,60,-0.135,-0.021
73,RV,2020-06-11,2020-07-09,C,-0.139,-0.498,0.234,18,-0.051,-0.171
85,RV,2020-06-11,2020-07-09,P,-0.188,-0.555,0.167,18,-0.009,-0.112
97,RV,2020-09-28,2020-09-30,C,-0.499,-1.052,0.101,3,0.065,-0.152
109,RV,2020-09-28,2020-09-30,P,0.044,-0.131,0.423,3,-0.094,-0.101


TSLA stress episodes, R2 over R1 (W12)


,definition,start,end,cp,gain,lo,hi,sessions,best_constant,fitted_constant
1,RV,2018-02-12,2018-03-05,C,0.030,-0.009,0.068,15,-0.084,-0.012
13,RV,2018-02-12,2018-03-05,P,-0.021,-0.119,0.081,15,-0.024,0.032
25,RV,2018-12-26,2019-01-23,C,0.247,-0.151,0.314,19,-0.203,-0.074
37,RV,2018-12-26,2019-01-23,P,0.113,-0.128,0.151,19,-0.148,-0.038
49,RV,2020-02-27,2020-05-21,C,-0.430,-0.643,-0.040,60,-0.094,0.106
61,RV,2020-02-27,2020-05-21,P,-0.585,-0.869,-0.034,60,-0.092,0.119
73,RV,2020-06-11,2020-07-09,C,0.113,0.064,0.129,18,0.357,0.021
85,RV,2020-06-11,2020-07-09,P,0.164,0.106,0.201,18,0.350,0.020
97,RV,2020-09-28,2020-09-30,C,0.076,-0.283,0.371,3,0.079,0.068
109,RV,2020-09-28,2020-09-30,P,0.078,-0.343,0.420,3,0.109,0.066


QQQ stress episodes, R2 over R1 (W12)


,definition,start,end,cp,gain,lo,hi,sessions,best_constant,fitted_constant
1,RV,2018-02-12,2018-03-05,C,0.242,0.006,0.373,15,-0.185,-0.112
13,RV,2018-02-12,2018-03-05,P,0.069,-0.223,0.271,15,-0.083,-0.093
25,RV,2018-12-26,2019-01-23,C,-0.002,-0.291,0.158,19,-0.061,-0.131
37,RV,2018-12-26,2019-01-23,P,-0.158,-0.548,0.123,19,0.019,-0.114
49,RV,2020-02-27,2020-05-21,C,0.212,-0.758,0.396,60,-0.272,-0.216
61,RV,2020-02-27,2020-05-21,P,0.083,-0.364,0.244,60,-0.156,-0.161
73,RV,2020-06-11,2020-07-09,C,-0.345,-0.762,0.001,18,-0.047,-0.262
85,RV,2020-06-11,2020-07-09,P,-0.126,-0.685,0.344,18,-0.082,-0.159
97,RV,2020-09-28,2020-09-30,C,-0.383,-0.607,0.242,3,0.103,-0.235
109,RV,2020-09-28,2020-09-30,P,-0.355,-0.444,0.195,3,0.489,-0.153


SLV stress episodes, R2 over R1 (W12)


,definition,start,end,cp,gain,lo,hi,sessions,best_constant,fitted_constant
1,RV,2018-02-12,2018-03-05,C,-0.061,-0.196,0.072,15,0.105,-0.059
13,RV,2018-02-12,2018-03-05,P,-0.017,-0.057,0.016,15,0.177,-0.010
25,RV,2018-12-26,2019-01-23,C,-0.005,-0.013,0.003,19,0.045,-0.061
37,RV,2018-12-26,2019-01-23,P,-0.017,-0.034,-0.002,19,0.140,-0.053
49,RV,2020-02-27,2020-05-21,C,-0.056,-0.357,-0.003,60,-0.370,-0.069
61,RV,2020-02-27,2020-05-21,P,-0.073,-0.494,-0.003,60,-0.392,-0.073
73,RV,2020-06-11,2020-07-09,C,-0.042,-0.104,0.028,18,-0.028,-0.244
85,RV,2020-06-11,2020-07-09,P,-0.174,-0.425,-0.049,18,0.128,-0.277
97,RV,2020-09-28,2020-09-30,C,0.081,-0.259,0.566,3,-0.069,-0.085
109,RV,2020-09-28,2020-09-30,P,-0.036,-0.333,0.382,3,-0.021,-0.067


In [8]:
for u in U:
    L = pd.read_csv(R / "stocks" / u / "level_shape_by_year.csv")
    print(u.upper(), "level (P0 over R1), shape (R2 over P0) and the ceiling, by year")
    display(L.pivot_table(index="year", columns="cp", values=["level_P0_over_R1", "shape_R2_over_P0", "ceiling_best_over_R1"]).round(3))
    P = pd.read_csv(R / "stocks" / u / "parameter_stability_summary.csv")
    display(P[(P.rule == "P0")].round(3))

AAPL level (P0 over R1), shape (R2 over P0) and the ceiling, by year


ceiling_best_over_R1        level_P0_over_R1        shape_R2_over_P0       
cp                      C      P                C      P                C      P
year                                                                            
2017                0.107  0.080           -0.054 -0.039            0.006  0.014
2018                0.182  0.204            0.095  0.119            0.019  0.017
2019                0.234  0.223            0.109  0.106           -0.003  0.014
2020                0.295  0.113            0.047  0.021           -0.003 -0.015
2021                0.043  0.053            0.014  0.011            0.017  0.004
2022                0.191  0.174            0.113  0.023            0.008  0.000
2023                0.060  0.046            0.002 -0.017            0.009  0.003

,window,cp,rule,coef,min,max,median_se,median_abs_change_over_se,share_abs_z12_gt2
0,12,C,P0,c,-0.274,0.090,0.030,0.288,0.321
5,12,P,P0,c,-0.167,0.045,0.033,0.271,0.167
10,36,C,P0,c,-0.226,-0.039,0.041,0.072,NaN
15,36,P,P0,c,-0.135,-0.022,0.043,0.058,NaN
20,60,C,P0,c,-0.133,-0.125,0.034,0.025,NaN
25,60,P,P0,c,-0.086,-0.072,0.036,0.021,NaN


NVDA level (P0 over R1), shape (R2 over P0) and the ceiling, by year


ceiling_best_over_R1        level_P0_over_R1        shape_R2_over_P0       
cp                      C      P                C      P                C      P
year                                                                            
2017                0.218  0.270           -0.308 -0.172           -0.007 -0.010
2018                0.469  0.426            0.232  0.231           -0.000  0.001
2019                0.118  0.120           -0.121 -0.146            0.001  0.003
2020                0.231  0.126           -0.013 -0.022           -0.011 -0.003
2021                0.116  0.140           -0.023 -0.024            0.002  0.000
2022                0.209  0.172           -0.016 -0.098            0.011  0.005
2023                0.443  0.473           -0.171 -0.089           -0.005 -0.006

,window,cp,rule,coef,min,max,median_se,median_abs_change_over_se,share_abs_z12_gt2
0,12,C,P0,c,-0.197,0.363,0.049,0.171,0.298
5,12,P,P0,c,-0.186,0.323,0.050,0.170,0.321
10,36,C,P0,c,-0.161,0.030,0.036,0.080,NaN
15,36,P,P0,c,-0.126,0.053,0.037,0.075,NaN
20,60,C,P0,c,-0.099,-0.025,0.035,0.077,NaN
25,60,P,P0,c,-0.069,-0.005,0.032,0.063,NaN


... (6 outputs left out)

,window,cp,rule,coef,min,max,median_se,median_abs_change_over_se,share_abs_z12_gt2
0,12,C,P0,c,-0.319,0.061,0.035,0.278,0.226
5,12,P,P0,c,-0.186,0.075,0.032,0.225,0.119
10,36,C,P0,c,-0.267,-0.036,0.049,0.048,NaN
15,36,P,P0,c,-0.164,-0.053,0.041,0.050,NaN
20,60,C,P0,c,-0.176,-0.130,0.044,0.018,NaN
25,60,P,P0,c,-0.127,-0.101,0.037,0.024,NaN


SLV level (P0 over R1), shape (R2 over P0) and the ceiling, by year


ceiling_best_over_R1        level_P0_over_R1        shape_R2_over_P0       
cp                      C      P                C      P                C      P
year                                                                            
2017                0.063  0.067            0.003 -0.056            0.019  0.015
2018                0.013  0.014            0.000 -0.000            0.002  0.002
2019                0.047  0.040           -0.030 -0.011           -0.000  0.001
2020                0.231  0.259           -0.118 -0.157           -0.002 -0.003
2021                0.186  0.190           -0.068 -0.043            0.001  0.000
2022                0.080  0.112           -0.002  0.016            0.002  0.016
2023                0.054  0.092            0.005 -0.003            0.001  0.004

,window,cp,rule,coef,min,max,median_se,median_abs_change_over_se,share_abs_z12_gt2
0,12,C,P0,c,-0.316,0.176,0.033,0.299,0.179
5,12,P,P0,c,-0.365,0.177,0.037,0.247,0.214
10,36,C,P0,c,-0.252,0.088,0.057,0.069,NaN
15,36,P,P0,c,-0.291,0.135,0.059,0.074,NaN
20,60,C,P0,c,-0.057,0.001,0.056,0.013,NaN
25,60,P,P0,c,-0.039,0.021,0.058,0.012,NaN


In [9]:
for u in U:
    M = pd.read_csv(R / "stocks" / u / "maturity_by_year.csv")
    print(u.upper(), "tau^(-alpha) against alpha = 0.5 (R2)")
    display(M.pivot_table(index=["sessions_kind", "year"], columns=["rule", "cp"], values="gain").round(3))

AAPL tau^(-alpha) against alpha = 0.5 (R2)


rule               alpha:0        alpha:0.25        alpha:0.75        alpha:1       
cp                       C      P          C      P          C      P       C      P
sessions_kind year                                                                  
all           2017   0.019  0.011      0.008  0.005      0.001 -0.001   0.006  0.001
              2018  -0.029 -0.060     -0.010 -0.025     -0.011 -0.004  -0.034 -0.026
              2019  -0.048 -0.052     -0.023 -0.023      0.004  0.001  -0.012 -0.014
              2020  -0.016  0.001     -0.007  0.001     -0.004 -0.005  -0.013 -0.010
              2021  -0.011 -0.003     -0.005 -0.000     -0.002 -0.003  -0.008 -0.008
              2022  -0.002  0.004      0.008  0.006     -0.035 -0.013  -0.075 -0.025
              2023  -0.010 -0.005     -0.004 -0.003      0.000  0.004  -0.003  0.006
calm          all    0.010 -0.009      0.002 -0.005      0.008  0.003   0.020  0.004
stress        all   -0.070 -0.004     -0.023  0.003     -0.033 -0.015  -0.097 -0.032

NVDA tau^(-alpha) against alpha = 0.5 (R2)


rule               alpha:0        alpha:0.25        alpha:0.75        alpha:1       
cp                       C      P          C      P          C      P       C      P
sessions_kind year                                                                  
all           2017   0.119  0.066      0.041  0.019      0.000  0.005   0.016  0.017
              2018  -0.133 -0.133     -0.060 -0.057      0.009  0.008  -0.020 -0.017
              2019   0.050  0.059      0.023  0.027     -0.006 -0.007   0.003  0.002
              2020   0.001 -0.008     -0.001 -0.007      0.006  0.009   0.014  0.016
              2021   0.004  0.024      0.001  0.012      0.003 -0.005   0.008 -0.002
              2022   0.007 -0.008      0.004 -0.010     -0.003  0.020  -0.005  0.038
              2023   0.012  0.030     -0.001  0.010      0.022  0.006   0.055  0.022
calm          all    0.008  0.006     -0.001 -0.000      0.012  0.006   0.027  0.014
stress        all    0.009 -0.000      0.006 -0.003     -0.007  0.007  -0.010  0.014

TSLA tau^(-alpha) against alpha = 0.5 (R2)


rule               alpha:0        alpha:0.25        alpha:0.75        alpha:1       
cp                       C      P          C      P          C      P       C      P
sessions_kind year                                                                  
all           2017   0.021  0.008      0.003  0.003      0.012  0.002   0.027  0.007
              2018   0.023  0.012      0.016  0.007     -0.017 -0.006  -0.028 -0.009
              2019   0.033  0.029      0.020  0.017     -0.019 -0.017  -0.029 -0.028
              2020  -0.002 -0.011     -0.011 -0.016      0.029  0.031   0.057  0.057
              2021  -0.000 -0.001     -0.000 -0.000      0.001 -0.001   0.003 -0.002
              2022  -0.018 -0.005     -0.009 -0.005      0.003  0.008  -0.001  0.014
              2023   0.031  0.060      0.012  0.028      0.004 -0.008   0.019  0.000
calm          all    0.002  0.005      0.002  0.005     -0.005 -0.012  -0.012 -0.026
stress        all   -0.001 -0.019     -0.019 -0.033      0.060  0.073   0.122  0.143

QQQ tau^(-alpha) against alpha = 0.5 (R2)


rule               alpha:0        alpha:0.25        alpha:0.75        alpha:1       
cp                       C      P          C      P          C      P       C      P
sessions_kind year                                                                  
all           2017  -0.010 -0.026      0.001 -0.007     -0.011 -0.006  -0.027 -0.019
              2018  -0.035 -0.038     -0.014 -0.013     -0.002 -0.005  -0.014 -0.021
              2019  -0.015 -0.018     -0.001 -0.006     -0.014 -0.005  -0.034 -0.017
              2020  -0.008 -0.023     -0.001 -0.012     -0.008  0.005  -0.021 -0.000
              2021  -0.034 -0.050     -0.017 -0.027      0.008  0.019   0.004  0.023
              2022  -0.012 -0.014     -0.001 -0.003     -0.008 -0.007  -0.016 -0.015
              2023   0.030  0.033      0.010  0.010      0.002  0.003   0.009  0.012
calm          all    0.001 -0.024     -0.004 -0.012      0.016  0.009   0.037  0.013
stress        all   -0.023 -0.023     -0.002 -0.011     -0.025  0.001  -0.065 -0.010

SLV tau^(-alpha) against alpha = 0.5 (R2)


rule               alpha:0        alpha:0.25        alpha:0.75        alpha:1       
cp                       C      P          C      P          C      P       C      P
sessions_kind year                                                                  
all           2017   0.017  0.016      0.010  0.008     -0.011 -0.003  -0.020 -0.001
              2018  -0.001 -0.002     -0.000 -0.001      0.000  0.001  -0.000  0.001
              2019   0.008  0.011      0.004  0.006     -0.002 -0.006  -0.000 -0.008
              2020   0.016  0.017      0.004  0.002      0.011  0.019   0.034  0.049
              2021  -0.007  0.003     -0.006  0.000      0.012  0.003   0.025  0.009
              2022   0.005 -0.007      0.002 -0.002     -0.001 -0.002   0.000 -0.006
              2023   0.010  0.005      0.005  0.003     -0.005 -0.004  -0.008 -0.007
calm          all    0.011  0.018      0.003  0.005      0.007  0.008   0.022  0.026
stress        all    0.001 -0.009     -0.003 -0.009      0.010  0.018   0.023  0.037

**Reading.** The spot–volatility relation exists within each period but does not carry forward reliably in these trailing 12-month fits. The level that fits each month best, after the fact, would have removed a median of 11–22% of the three stocks' yearly error (over 2017–2023, by stock and option type; single years run from 4% to 47%): the ceiling column. The level fitted on the previous 12 months gains little: its median yearly Gain is +0.047 / +0.021 for AAPL and negative for NVDA (−0.023 / −0.089) and TSLA (−0.046 / −0.038), calls / puts, though the level column is positive in some years (for example NVDA in 2018).

These trailing 12-month fits recover little of the three stocks' after-the-fact ceiling. For comparison, SPX's trailing level reached at least half of its ceiling in 15 of 26 type-years (2011–2023); it fell short in 2017–2019, 2022–2023 and for 2012's calls.

## Books, data starvation and held positions on the five (§16.11, §16.13, §16.14)

In [10]:
for u in U:
    d = R / "extensions" / u
    if (d / "books_median_yearly_gain.csv").exists():
        m = pd.read_csv(d / "books_median_yearly_gain.csv")
        print(u.upper(), "books, median yearly Gain over R1")
        display(m[(m.base == "R1") & m.rule.isin(["R2", "P0", "R4"])].pivot_table(index=["book", "rule"], columns="measure", values="median_yearly_gain").round(3))
    if (d / "starve_seed_spread.csv").exists():
        sp = pd.read_csv(d / "starve_seed_spread.csv")
        print(u.upper(), "starvation, median yearly Gain of R2 over R1")
        display(sp[(sp.base == "R1") & (sp.rule == "R2")].pivot_table(index="config", columns="cp", values="median").round(3))
    if (d / "paths_gains.csv").exists():
        G = pd.read_csv(d / "paths_gains.csv")
        print(u.upper(), "held positions, R2 over R1, pooled periods")
        display(G[(G.subset == "all") & (G.base == "R1") & (G.rule == "R2") & ~G.period.str.fullmatch(r"\d{4}")]
                .pivot_table(index=["H", "period"], columns=["measure", "cp"], values="gain").round(3))

AAPL books, median yearly Gain over R1


measure              book  per_option
book          rule                   
calls         P0    0.069       0.047
              R2    0.060       0.044
              R4    0.046      -0.010
long_vol      P0    0.081       0.063
              R2    0.062       0.063
              R4    0.077       0.052
puts          P0    0.018       0.021
              R2    0.016       0.015
              R4    0.043       0.019
risk_reversal P0    0.136       0.063
              R2    0.226       0.063
              R4   -0.076       0.052

AAPL starvation, median yearly Gain of R2 over R1


cp,C,P
config,,
W1,0.020,0.008
W12,0.044,0.015
W2,0.015,-0.007
W3,0.053,0.002
W6,0.059,0.007
frac0.01,0.040,0.013
frac0.05,0.048,0.010
frac0.25,0.044,0.014
k10,0.040,0.014


AAPL held positions, R2 over R1, pooled periods


measure      one_session          path       
cp                     C      P      C      P
H  period                                    
5  2017-2019       0.087  0.100  0.045  0.104
   2021-2023       0.048 -0.001  0.053  0.018
10 2017-2019       0.095  0.106 -0.035  0.018
   2021-2023       0.046 -0.010 -0.004 -0.058

... (25 outputs left out)

 starvation, median yearly Gain of R2 over R1


cp,C,P
config,,
W1,-0.034,-0.057
W12,-0.000,-0.010
W2,-0.026,-0.030
W3,-0.006,-0.033
W6,-0.034,-0.009
frac0.01,-0.007,-0.049
frac0.05,-0.002,-0.012
frac0.25,-0.002,-0.012
k10,-0.003,-0.018


SLV held positions, R2 over R1, pooled periods


measure      one_session          path       
cp                     C      P      C      P
H  period                                    
5  2017-2019      -0.002  0.002  0.013 -0.003
   2021-2023      -0.043 -0.024 -0.022 -0.013
10 2017-2019      -0.001  0.001  0.002 -0.025
   2021-2023      -0.048 -0.031 -0.081 -0.121

**Reading.** Books amplify whatever the sign: gains for QQQ (calls 0.162 on the book, against 0.083 per option) and AAPL, losses for NVDA and TSLA.

The risk-reversal book gains 0.22–0.24 for AAPL, TSLA and SLV. That is unlike the indices, where it is the control. One possible reason, not investigated, is that out-of-the-money calls and puts on these underlyings may not share one volatility move.

QQQ's Gain grows with the holding period in 2017–2019 (10 sessions: 0.333 / 0.275, against 0.122 / 0.102 one session), as SPX's does. The stocks are mixed. Most of the five lose on paths in 2020 and 2021–2023.

On every underlying, the path code's first-day errors equal the walk-forward's exactly, and 79–94% of eligible paths complete.